# Lab 1 – Supervised Learning

## Dataset description - CIC-IDS2017

The dataset was obtained in the following URL (https://cicresearch.ca/CICDataset/CIC-IDS-2017/download.php). However, it is necessary to register in order to be able to download it.

- **Source:** Canadian Institute for Cybersecurity, University of New Brunswick.
- **Official URL:** https://www.unb.ca/cic/datasets/ids-2017.html
- **Date of Download:** 11-09-2026
- **License:** <cite>The CICIDS2017 dataset consists of labeled network flows, including full packet payloads in pcap format, the corresponding profiles and the labeled flows (GeneratedLabelledFlows.zip) and CSV files for machine and deep learning purpose (MachineLearningCSV.zip) are publicly available for researchers.</cite>


## Part A – Data acquisition and hygiene
### Unzip contents

In [ ]:
# !unzip -n ../data/MachineLearningCSV.zip -d ../data/

### Load data on CSV, strip column names, inspect shape/dtypes/labels

In [ ]:
import glob
import os

import numpy as np
import pandas as pd

DATA_PATH = os.path.join("../data", "MachineLearningCVE")
all_files = glob.glob(os.path.join(DATA_PATH, "*.csv"))

df = pd.concat((pd.read_csv(f, low_memory=False) for f in all_files), ignore_index=True)

# Strip whitespace from column names e.g. ' Label' -> 'Label'
df.columns = df.columns.str.strip()

print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes.value_counts())
print("\nLabel value counts:\n", df["Label"].value_counts())

df.head()

### Audit Inf / NaN rates and apply a written imputation rule

In [ ]:
# Replace infinites with NaN so they show up in the same audit
df.replace([np.inf, -np.inf], np.nan, inplace=True)

nan_counts = df.isna().sum()
nan_report = nan_counts[nan_counts > 0].sort_values(ascending=False)
print("Columns with NaN/Inf (post-replacement):\n", nan_report)
print(f"\nTotal rows: {len(df)}")
print(
    f"Rows with at least one NaN: {df.isna().any(axis=1).sum()} "
    f"({df.isna().any(axis=1).mean() * 100:.3f}%)"
)

After reviewing the number of NaN values, we can see that it is only around **$0.1\%$** of the total dataset. This percentage is not relevant and can be dropped.

In [ ]:
row_nan_frac = df.isna().any(axis=1).mean()

low_nan_cols = nan_report[nan_report / len(df) < 0.005].index.tolist()
high_nan_cols = nan_report[nan_report / len(df) >= 0.005].index.tolist()

print("Columns to handle by row-drop:", low_nan_cols)
print("Columns to handle by train-median imputation later:", high_nan_cols)

# Drop rows only for the low-NaN columns now (safe, minimal loss)
if low_nan_cols:
    before = len(df)
    df = df.dropna(subset=low_nan_cols)
    print(
        f"Dropped {before - len(df)} rows due to low-frequency NaNs in {low_nan_cols}"
    )

### Leakage hunt: direct label encoders, duplicate row, label typos

The data leakage could happen when we keep data that could be found as well on the test, validation or target dataset. In this case, we need to check which information could be found on the target dataset as well and avoid feeding it to the model as it could influence the classification.

#### Duplicate rows

In [ ]:
n_duplicates = df.duplicated().sum()
print(f"Exact duplicate rows: {n_duplicates} ({n_duplicates / len(df) * 100:.3f}%)")
df = df.drop_duplicates()
print(f"Shape after dropping duplicates: {df.shape}")

#### Duplicate / near-duplicate columns

In [ ]:
duplicated_cols = df.columns[df.columns.duplicated()].tolist()
print("Duplicate column names:", duplicated_cols)
# Also check columns that are byte-for-byte identical to another column
identical_pairs = []
cols = df.select_dtypes(include=[np.number]).columns
for i, c1 in enumerate(cols):
    for c2 in cols[i + 1 :]:
        if df[c1].equals(df[c2]):
            identical_pairs.append((c1, c2))
print(
    "Identical numeric column pairs (candidates to drop one of each):", identical_pairs
)


#### Label typos / inconsistent casing

In [ ]:
print("\nRaw unique label strings:", df["Label"].unique())
df["Label"] = df["Label"].str.strip()  # trailing/leading whitespace in label strings
print("Unique labels after stripping whitespace:", df["Label"].unique())

#### Columns that could encode the label directly

In [ ]:
leakage_candidate_cols = [
    c
    for c in [
        "Flow ID",
        "Source IP",
        "Destination IP",
        "Timestamp",
        "Src IP",
        "Dst IP",
        "SimillarHTTP",
    ]
    if c in df.columns
]
print(
    "\nIdentifier / leakage-risk columns to exclude from features:",
    leakage_candidate_cols,
)

### Map labels to binary

In [ ]:
# Binary mapping: 0 = BENIGN, 1 = any attack
df["Label"] = df["Label"].replace(
    {"Benign": "BENIGN"}
)  # normalize casing found in step 4c

df["y"] = (df["Label"] != "BENIGN").astype(int)

print(df["y"].value_counts())
print(f"\nAttack ratio: {df['y'].mean() * 100:.3f}%")

# Keep the original multi-class label around for later error analysis (Part D, task 18)
y_multiclass = df["Label"].copy()

## Part B - Features and split
### Select numeric features only, document exclusions

In [ ]:
excluded_cols = leakage_candidate_cols + ["Label", "y"]
excluded_cols = [c for c in excluded_cols if c in df.columns]

feature_df = df.drop(columns=excluded_cols)
numeric_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_dropped = [c for c in feature_df.columns if c not in numeric_cols]

print(f"Numeric features kept ({len(numeric_cols)}):\n{numeric_cols}")
print(
    f"\nNon-numeric columns excluded ({len(non_numeric_dropped)}): {non_numeric_dropped}"
)

**Excluded columns and why:**

| Column(s) | Reason for exclusion |
|---|---|
| `Flow ID`, `Source IP`, `Destination IP`, `Timestamp`, `Src IP`, `Dst IP` | Identifiers — risk of the model memorizing specific hosts/times instead of learning traffic behavior (leakage / poor generalization). |
| `Label` | Target variable, not a feature. |
| Any duplicate/identical column found in 4b | Redundant, adds no signal, slightly biases importance ranking. |
| Non-numeric columns (if any remain) | Random Forest here is trained on numeric flow statistics only, per lab scope; categorical encodings are out of scope for this lab. |

### Split (time-based preferred) then scale on train only

In [ ]:
from sklearn.model_selection import train_test_split

X = feature_df[numeric_cols].copy()
y = df["y"].copy()


HAS_TIME_COLUMN = "Timestamp" in df.columns

if HAS_TIME_COLUMN:
    order = (
        df["Timestamp"].argsort()
        if pd.api.types.is_numeric_dtype(df["Timestamp"])
        else pd.to_datetime(df["Timestamp"], errors="coerce").argsort()
    )
    X_sorted, y_sorted = X.iloc[order], y.iloc[order]
    split_idx = int(len(X_sorted) * 0.7)
    X_train, X_test = X_sorted.iloc[:split_idx], X_sorted.iloc[split_idx:]
    y_train, y_test = y_sorted.iloc[:split_idx], y_sorted.iloc[split_idx:]
    split_method = "time-based (70% earliest / 30% latest)"
else:
    # Fallback: stratified random split, fixed seed for reproducibility
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=42
    )
    split_method = "stratified random 70/30, random_state=42"

print(f"Split method used: {split_method}")
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

**Split justification:** A time-based split is preferred for intrusion detection because it mirrors
real operational deployment — the model is trained on past traffic and evaluated on traffic it has
never seen chronologically, which avoids inflated metrics from near-duplicate flows of the same
attack burst leaking across train/test. Where a reliable timestamp ordering isn't available, we fall
back to a stratified random 70/30 split with a fixed `random_state=42` for reproducibility, preserving
the (heavily imbalanced) attack/benign ratio in both splits.

In [ ]:
# Median imputation for high-NaN columns, fit on TRAIN ONLY
if high_nan_cols:
    train_medians = X_train[high_nan_cols].median()
    X_train[high_nan_cols] = X_train[high_nan_cols].fillna(train_medians)
    X_test[high_nan_cols] = X_test[high_nan_cols].fillna(
        train_medians
    )  # reuse train medians

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

#### Class balance in train vs test

In [ ]:
train_balance = y_train.value_counts(normalize=True).rename("train_fraction")
test_balance = y_test.value_counts(normalize=True).rename("test_fraction")

balance_report = pd.concat([train_balance, test_balance], axis=1)
balance_report.index = ["BENIGN (0)", "ATTACK (1)"]
print(balance_report)

## Part C - Random Forest in depth

### Random Forest — model statement

A Random Forest is an **ensemble of B decision trees** trained on bootstrap-resampled subsets of the
training data. For classification, each tree $T_b$ produces a predicted class (or class-probability
vector) for an input $x$. The forest's final prediction is obtained by **majority vote** across trees:

$$\hat{y}(x) = \text{mode}\{T_1(x), T_2(x), \dots, T_B(x)\}$$

or, when probability estimates are needed (as in this lab, for thresholding), by **averaging the
per-tree class probabilities**:

$$\hat{p}(y=1 \mid x) = \frac{1}{B}\sum_{b=1}^{B} p_b(y=1 \mid x)$$

Each individual tree is grown deep (often to near-purity) and is therefore high-variance / low-bias;
averaging many such trees trained on different resamples of the data cancels out much of that variance
while keeping the low bias, which is the central idea behind bagging-based ensembles.

### Bootstrap, feature randomness, and variance reduction: Why Random Forest reduces variance vs. a single deep tree

- **Bootstrap samples (bagging):** Each of the $B$ trees is trained on a random sample of the
  training rows, drawn *with replacement* and the same size as the original training set. On
  average, each bootstrap sample contains about 63.2% of the unique original rows (the rest are
  duplicates), leaving out ~36.8% ("out-of-bag" rows) that could optionally be used to estimate
  generalization error without a separate validation set.
- **Feature randomness (`max_features`):** At each split in each tree, only a random subset of the
  features (e.g. $\sqrt{p}$ features for classification, controlled by `max_features`) is considered
  as split candidates, instead of all $p$ features. This decorrelates the trees — without it, most
  trees would keep splitting on the same few dominant features (e.g. `Flow Duration`,
  `Destination Port`) and their errors would be highly correlated, limiting the variance-reduction
  benefit of averaging.
- **Variance reduction:** A single deep decision tree tends to overfit — it has low bias but high
  variance, meaning small changes in the training data produce very different trees. Averaging $B$
  trees whose errors are only weakly correlated (thanks to bagging + feature randomness) reduces the
  variance of the ensemble roughly by a factor related to their average pairwise correlation, without
  increasing bias much — formally, for trees with variance $\sigma^2$ and average pairwise
  correlation $\rho$: $\text{Var(average)} = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2(1-\rho)$,
  which shrinks toward $\rho\sigma^2$ as $B$ grows — this is why lowering $\rho$ (via feature
  randomness) matters as much as increasing $B$.

### Train baseline Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_baseline = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)

rf_baseline.fit(X_train_scaled, y_train)

print("Baseline RF trained.")
print("OOB not enabled (set oob_score=True for an out-of-bag estimate if desired).")

### Hyperparameter tuning

In [ ]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": randint(100, 400),
    "max_depth": [10, 15, 20, 30, None],
    "max_features": ["sqrt", "log2", 0.3, 0.5],
    "min_samples_leaf": randint(1, 6),
}

base_estimator = RandomForestClassifier(
    class_weight="balanced", n_jobs=-1, random_state=42
)

# Use a modest sample of the training set for the search to keep runtime reasonable,
# then refit the best params on the full training set.
search = RandomizedSearchCV(
    estimator=base_estimator,
    param_distributions=param_dist,
    n_iter=15,
    scoring="average_precision",  # PR-AUC — appropriate under class imbalance
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train_scaled, y_train)

print("Best params found:", search.best_params_)
print("Best CV average precision:", search.best_score_)

rf_tuned = search.best_estimator_

In [ ]:
import os

import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(rf_baseline, "models/rf_baseline.joblib")
joblib.dump(rf_tuned, "models/rf_tuned.joblib")
joblib.dump(scaler, "models/scaler.joblib")

print("Saved: rf_baseline.joblib, rf_tuned.joblib, scaler.joblib")

### Threshold selection: precision/recall vs threshold table

In [ ]:
from sklearn.metrics import precision_recall_curve

y_proba = rf_tuned.predict_proba(X_test_scaled)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

# Sample at least 5 thresholds across the range for a readable table
sample_thresholds = np.linspace(0.1, 0.9, 5)
rows = []
for t in sample_thresholds:
    idx = np.searchsorted(thresholds, t)
    idx = min(idx, len(precisions) - 1)
    y_pred_t = (y_proba >= t).astype(int)
    tp = ((y_pred_t == 1) & (y_test == 1)).sum()
    fp = ((y_pred_t == 1) & (y_test == 0)).sum()
    fn = ((y_pred_t == 0) & (y_test == 1)).sum()
    tn = ((y_pred_t == 0) & (y_test == 0)).sum()
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    rows.append(
        {
            "threshold": t,
            "precision": precision,
            "recall": recall,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
        }
    )

threshold_table = pd.DataFrame(rows)
print(threshold_table)

# Chosen operating threshold (justify in markdown below based on the printed table)
CHOSEN_THRESHOLD = (
    0.3  # example — adjust based on the SOC priority discussed in Part D task 19
)

**Threshold choice rationale:** We do not use the default 0.5 cutoff because, under class imbalance
and a security use case, false negatives (missed attacks) are typically costlier than false positives
(extra alerts an analyst must triage). Based on the table above, threshold **`<FILL IN CHOSEN VALUE>`**
was selected because it <FILL IN — e.g. "keeps recall on the attack class above 0.9 while keeping
precision high enough to avoid overwhelming the SOC queue">.

### Feature importance: top 15

In [ ]:
import matplotlib.pyplot as plt

importances = pd.Series(rf_tuned.feature_importances_, index=X_train_scaled.columns)
top15 = importances.sort_values(ascending=False).head(15)


plt.figure(figsize=(8, 6))
top15.sort_values().plot(kind="barh")
plt.xlabel("Gini importance")
plt.title("Top 15 feature importances — Random Forest")
plt.tight_layout()
plt.show()

print(top15)

**Security reading of top features (fill in with your actual top-15 list):**

- **Flow duration / inter-arrival time features** (e.g. `Flow Duration`, `Flow IAT Mean`) often rank
  highly because many automated attacks (port scans, DoS floods) produce flows with very short,
  regular durations compared to human-driven benign sessions.
- **Packet-rate features** (e.g. `Flow Packets/s`, `Fwd Packets/s`) are informative because
  volumetric attacks (DoS/DDoS) push packet rates far outside normal traffic envelopes.
- **TCP flag counts** (e.g. `SYN Flag Count`, `ACK Flag Count`) matter because scanning and certain
  DoS variants (SYN flood) produce abnormal flag distributions (e.g. many SYNs with few completed
  handshakes).
- Note: we deliberately avoid describing exact numeric thresholds an attacker could use to stay under
  detection ("attack recipes") — the discussion stays at the level of *which behavioral signal* the
  model relies on, not exploitable cutoff values.

## Part D - Evaluation and failure analysis
### Confusion matrix, precision, recall, FPR, F1, ROC-AUC, PR-AUC

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

y_pred = (y_proba >= CHOSEN_THRESHOLD).astype(int)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
fpr = fp / (fp + tn)
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

metrics_table = pd.DataFrame(
    [
        {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "FPR": fpr,
            "ROC-AUC": roc_auc,
            "PR-AUC": pr_auc,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
        }
    ]
)
print(metrics_table.round(4))

ConfusionMatrixDisplay(cm, display_labels=["BENIGN", "ATTACK"]).plot(cmap="Blues")
plt.title(f"Confusion matrix @ threshold={CHOSEN_THRESHOLD}")
plt.show()

In [ ]:
import json
from datetime import datetime

bundle = {
    "model": rf_tuned,
    "scaler": scaler,
    "feature_columns": list(X_train_scaled.columns),
    "chosen_threshold": CHOSEN_THRESHOLD,
    "best_params": search.best_params_,
    "random_state": 42,
    "trained_on": DATA_PATH,
    "timestamp": datetime.now().isoformat(),
}

joblib.dump(bundle, "models/rf_cicids2017_bundle.joblib")

# Human-readable metadata alongside it, for the report / reproducibility section
metadata = {k: v for k, v in bundle.items() if k not in ("model", "scaler")}
with open("models/rf_cicids2017_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print("Saved deployment bundle + metadata.")

### Dummy baseline comparison

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train_scaled, y_train)
y_pred_dummy = dummy.predict(X_test_scaled)

dummy_recall = recall_score(y_test, y_pred_dummy, zero_division=0)
dummy_precision = precision_score(y_test, y_pred_dummy, zero_division=0)
dummy_f1 = f1_score(y_test, y_pred_dummy, zero_division=0)

comparison = pd.DataFrame(
    [
        {
            "model": "Dummy (always-BENIGN)",
            "precision": dummy_precision,
            "recall": dummy_recall,
            "f1": dummy_f1,
        },
        {
            "model": "Random Forest (tuned)",
            "precision": precision,
            "recall": recall,
            "f1": f1,
        },
    ]
)
print(comparison)

**Baseline comparison:** The always-BENIGN dummy baseline achieves 0 recall on the attack class by
construction (it never predicts "attack"), while overall "accuracy" can still look deceptively high
under class imbalance — this is exactly why accuracy alone is not reported as a success metric in
this lab. The tuned Random Forest must show a **meaningful recall improvement on the attack class**
over this baseline to be considered a working detector (see metrics table above for the actual gap).

### Five failure cases

In [ ]:
results_df = X_test.copy()
results_df["y_true"] = y_test.values
results_df["y_pred"] = y_pred
results_df["y_proba"] = y_proba
results_df["true_label_detail"] = y_multiclass.loc[
    X_test.index
].values  # original multi-class label

false_positives = results_df[(results_df.y_true == 0) & (results_df.y_pred == 1)]
false_negatives = results_df[(results_df.y_true == 1) & (results_df.y_pred == 0)]

print(f"Total FP: {len(false_positives)}, Total FN: {len(false_negatives)}")

# Sample up to 3 FPs and 2 FNs (or adjust split) for the five-case write-up
sample_fp = false_positives.sample(min(3, len(false_positives)), random_state=42)
sample_fn = false_negatives.sample(min(2, len(false_negatives)), random_state=42)

# Show only a compact, sanitized set of columns (drop anything identifier-like, already excluded)
display_cols = list(top15.index[:6]) + [
    "y_true",
    "y_pred",
    "y_proba",
    "true_label_detail",
]
print("\n--- Sample False Positives ---")
print(sample_fp[display_cols])
print("\n--- Sample False Negatives ---")
print(sample_fn[display_cols])

**Five failure cases (fill in using the printed rows above):**

1. **[FP] Row index `<...>`** — predicted ATTACK, true BENIGN. Feature values: `<summarize top
   features for this row>`. Likely cause: `<e.g. an unusually short/bursty but legitimate flow,
   such as a benign health-check or retry storm, that resembles scan-like timing>`.
2. **[FP] Row index `<...>`** — `<same structure>`.
3. **[FP] Row index `<...>`** — `<same structure>`.
4. **[FN] Row index `<...>`** — predicted BENIGN, true ATTACK (`<attack subtype from
   true_label_detail>`). Likely cause: `<e.g. a low-and-slow variant of the attack whose flow
   statistics sit inside the normal traffic envelope, or an attack subtype underrepresented in the
   training split>`.
5. **[FN] Row index `<...>`** — `<same structure>`.

General pattern observed: `<e.g. "most FNs come from attack subtype X, suggesting the training set
under-represents that subtype relative to test" — connect back to the class-balance table from
Part B task 9>`.

### SOC cost discussion: operational cost of the threshold choice

- **Lowering the threshold (more alerts):** Recall on the attack class increases (fewer attacks slip
  through), but precision drops — more benign flows get flagged. In a real SOC this translates
  directly into **analyst alert fatigue**: a high false-positive volume can cause genuine alerts to
  get triaged more slowly or dismissed outright ("cry wolf" effect), and it increases staffing cost
  or requires additional automated triage/enrichment tooling.
- **Raising the threshold (fewer alerts):** Precision improves and the SOC queue is more manageable,
  but recall drops — some real attacks (especially lower-confidence or novel variants) go undetected.
  The operational risk shifts from "analyst time wasted" to "breach dwell time" — an attack that
  isn't flagged can persist and escalate (lateral movement, exfiltration) before any other control
  catches it.
- **Practical takeaway:** The right threshold is a business/risk decision, not a purely statistical
  one — it depends on analyst headcount, the cost of a missed breach for this specific organization,
  and whether downstream automated response (e.g. auto-quarantine) exists to absorb false positives
  without human review. This is why we tabulated multiple thresholds (task 14) instead of reporting
  a single fixed operating point.

## Conclusion

- Summarize final chosen threshold and headline metrics (precision/recall/F1/PR-AUC) vs. the dummy
  baseline.
- State the top 3–5 features driving detections and what they suggest behaviorally.
- Note the biggest failure pattern from task 18 and what it implies for future work (e.g. more
  training data for underrepresented attack subtypes, additional features, ensemble with a
  time-series/behavioral model for low-and-slow attacks).
- Reiterate reproducibility info: `random_state=42` used throughout, package versions pinned in
  `requirements.txt`, and the exact CSV file(s)/paths used listed in the dataset card (task 1).